# Baselines

In this notebook we create three non-ML baselines for comparison.

## Random Recommender

A simple random recommender simply recommends 10 random items to every user.

In [1]:
from recs import *

from recs.metrics import *
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm; tqdm.pandas()

data = DataLoader(data_size='large')  # all sequence lengths = 25
histories, imprs, labels = data.init_padded_behaviors(data.test)

Available device: GPU 



In [2]:
# Simple check
data.news_titles.numpy()

array([b'',
       b'The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By',
       b'Walmart Slashes Prices on Last-Generation iPads', ...,
       b"Overindulge in Delicious Snacks, Meals, and Treats on the World's Most Epicurean Streets",
       b"'The Walking Dead' star Josh McDermitt says The Whisperers should be worried about Negan",
       b'Baseball history unpacked, November 8'], dtype=object)

### Random click prediction

In [3]:
# random one-click labels
random_preds = np.zeros_like(imprs, dtype=np.float32)
for i, imp in enumerate(imprs):
    # if np.any(imp):
    mask = np.where(imp != 0)[0] # ignore not padding
    random_click = np.random.choice(mask)
    random_preds[i, random_click] = 1
    
random_preds

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [4]:
pd.DataFrame(get_metrics(labels, random_preds), ['random'])

,auc,mean_mrr,ndcg@5,ndcg@10
random,0.5,0.1869,0.1758,0.2494


In [5]:
pd.DataFrame(
    get_norm_metrics(labels, random_preds, imprs, dataset_size='large'), index=['random'])

,mean_epc,mean_intra_list_diversity,mean_surprisal,prediction_coverage
random,1.3322,0.2455,8.252,0.5218


### Random ranking

This creates fixed-length arrays with random probabilities and evaluates them as recommendations. These provide a ceiling for diversity and novelty.

In [6]:
random_probs = np.zeros_like(imprs, dtype=np.float32)
for i, imp in enumerate(imprs):
    random_prob = np.random.rand(len(imp))
    random_probs[i] = (1. / np.sum(random_prob)) * random_prob

log_results(labels, random_probs, imprs, "Random Recommender", dataset_size='large')

Saved model predictions here: ../.data/Random_Recommender.npy


,Random Recommender
modelname,Random Recommender
dataset_size,large
timestamp,2025-03-28 14:24
auc,0.4988
mean_mrr,0.221
ndcg@5,0.2272
ndcg@10,0.294
mean_epc,1.8579
mean_intra_list_diversity,0.2517
mean_surprisal,11.3558



---

## MSN Recommender

We take the existing candidate news items, assign an order to them (high to low), and evaluate them.

In [7]:
one = np.linspace(start=1, stop=0, num=data.max_imps)
msn_rank = np.tile(one, (labels.shape[0],1))
msn_rank.shape

(251821, 25)

In [8]:
log_results(labels, msn_rank, imprs, "MSN Recommender", dataset_size='large')

Saved model predictions here: ../.data/MSN_Recommender.npy


,MSN Recommender
modelname,MSN Recommender
dataset_size,large
timestamp,2025-03-28 14:24
auc,0.5949
mean_mrr,0.2215
ndcg@5,0.2278
ndcg@10,0.2948
mean_epc,1.3797
mean_intra_list_diversity,0.246
mean_surprisal,8.564



---

## Popularity Recommender

Instead of creating individual user profiles, we create one hypothetical one. We take the 50 most popular (in clicks), embed those with BERT (sentence embeddings), and rank the candidates on similarity. 

In [9]:
pos_candidates = []
for imp, lab in zip(imprs, labels):
    mask = np.where(lab == 1)
    pos_candidates.extend(imp[mask])
# pos_candidates

In [10]:
pop = Counter(pos_candidates)
pop_50 = pop.most_common(50)
pop_50 = list(zip(*pop_50))[0]
top_50 = np.array(pop_50)

In [11]:
# Inspect most popular
news_titles = np.array(data.news_titles)
print(news_titles[top_50])

[b'Opinion: Colin Kaepernick is about to get what he deserves: a chance'
 b'South Carolina teen gets life in prison for deadly elementary school shooting'
 b'30 Best Black Friday Deals from Costco'
 b'Meghan Markle and Hillary Clinton Secretly Spent the Afternoon Together at Frogmore Cottage'
 b'This was uglier than a brawl. And Myles Garrett deserves suspension for rest of year after helmet attack.'
 b"The Real Reason McDonald's Keeps the Filet-O-Fish on Their Menu"
 b'3 Indiana judges suspended after a night of drinking turned into a White Castle brawl'
 b"Some believe Mason Rudolph, hit in head with his own helmet, isn't getting enough blame"
 b"Report: Police investigating woman's death after Redskins' player Montae Nicholson took her to hospital"
 b"Taylor Swift Rep Hits Back at Big Machine, Claims She's Actually Owed $7.9 Million in Unpaid Royalties"
 b'Police find 26 children behind false wall at Colorado day care'
 b'66 Cool Tech Gifts Anyone Would Be Thrilled to Receive'
 b"13

In [12]:
embedding_matrix = np.load(data_folder/'embedding_matrix_large_cls.npy')
# embedding_matrix = np.load(data_folder/'embedding_matrix_small_cls.npy')

In [13]:
# Get popular vector
popularity_vector = np.mean(embedding_matrix[top_50], axis=0)

# now for every session we rank the candidates on their similarity to the average popularity embedding
cand_by_pop_rankings = []
for imp in tqdm(imprs):
    user_embeddings = embedding_matrix[imp]
    cand_by_pop_ranking = cosine_similarity([popularity_vector], user_embeddings)[0]
    cand_by_pop_rankings.append(cand_by_pop_ranking)

  0%|          | 0/251821 [00:00<?, ?it/s]

In [14]:
log_results(labels, 
            np.array(cand_by_pop_rankings), 
            imprs, "Popularity Recommender", 
            dataset_size='large')

Saved model predictions here: ../.data/Popularity_Recommender.npy


,Popularity Recommender
modelname,Popularity Recommender
dataset_size,large
timestamp,2025-03-28 14:26
auc,0.5355
mean_mrr,0.2443
ndcg@5,0.2585
ndcg@10,0.3177
mean_epc,1.334
mean_intra_list_diversity,0.1552
mean_surprisal,8.2623


In [15]:
# np.save('pop_recommender.npy', cand_by_pop_rankings)
# np.save('random_recommender.npy', random_labels)
# np.save('.data/random_recommender_probs.npy', random_probs)